In [ ]:
# ==============================================================================
# RoadEye Phase 3: RESUME TRAINING SCRIPT (For New Colab Instances)
# Automatically downloads dataset, audits it, and resumes from last.pt
# ==============================================================================
# 3. Install & Import Dependencies
!pip install ultralytics roboflow
import os
import shutil
import glob
from google.colab import drive
from roboflow import Roboflow
from ultralytics import YOLO
from PIL import Image

# ── 1. STORAGE (Mounting Drive) ───────────────────────────────────────────────
drive.mount('/content/drive', force_remount=True)
drive_save_path = '/content/drive/MyDrive/RoadEye_V4_Final_Weights'
print("✅ Drive mounted.")

# Verify that the shortcut/folder exists on this new account
last_weights_path = os.path.join(drive_save_path, 'v4_native_audit_run', 'weights', 'last.pt')
if not os.path.exists(last_weights_path):
    raise FileNotFoundError(f"❌ ERROR: Cannot find weights at {last_weights_path}. Did you share/copy the folder to this new Drive?")
print("✅ Found last.pt weights! Ready to resume.")

# ── 2. DATASET DOWNLOAD (Needed on the new Colab server) ──────────────────────
# ⚠️ Insert your V4 API Key
rf      = Roboflow(api_key="RQMPMcpDeL2nFd04kTvI")
project = rf.workspace("roadeye-workspace").project("roadeye-josak")
version = project.version(4)
dataset = version.download("yolov8")

train_labels_dir = os.path.join(dataset.location, "train", "labels")
train_images_dir = os.path.join(dataset.location, "train", "images")

yaml_files   = glob.glob(f"{dataset.location}/**/*.yaml", recursive=True)
dataset_yaml = yaml_files[0]

# ── 3. DATA AUDIT (Re-cleaning to match original training exactly) ────────────
print("\n🔍 Running Data Audit to prepare clean environment...")
removed_count = 0
total_count   = 0

for label_file in os.listdir(train_labels_dir):
    if not label_file.endswith(".txt"): continue
    lbl_path = os.path.join(train_labels_dir, label_file)

    img_path = os.path.join(train_images_dir, label_file.replace(".txt", ".jpg"))
    if not os.path.exists(img_path):
        img_path = img_path.replace(".jpg", ".png")
    if not os.path.exists(img_path): continue

    with Image.open(img_path) as img:
        w_px, h_px = img.size

    with open(lbl_path, 'r') as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]

    clean_lines = []
    for line in lines:
        parts = line.split()
        if len(parts) != 5: continue
        total_count += 1
        cls, cx, cy, w, h = int(parts[0]), *map(float, parts[1:])
        box_area_px = (w * w_px) * (h * h_px)

        if box_area_px >= 100.0:
            clean_lines.append(line)
        else:
            removed_count += 1

    with open(lbl_path, 'w') as f:
        f.write("\n".join(clean_lines) + ("\n" if clean_lines else ""))

print(f"✅ Audit done: {removed_count} micro-boxes removed / {total_count} total")

# ── 4. RESUME TRAINING ────────────────────────────────────────────────────────
print(f"\n🚀 RESUMING Phase 3 Training from {last_weights_path}...")

# Load the model from the last saved checkpoint
model = YOLO(last_weights_path)

# Call train with resume=True.
# We pass data=dataset_yaml just in case the Colab path changed (e.g., RoadEye-5 instead of RoadEye-4)
model.train(data=dataset_yaml, resume=True)

print(f"\n✅ Training fully complete. Weights at: {drive_save_path}/v4_native_audit_run/weights/")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 144.3 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings w


Extracting Dataset Version Zip to RoadEye-4 in yolov8:: 100%|██████████| 6451/6451 [00:01<00:00, 4337.19it/s]



🔍 Running Data Audit to prepare clean environment...
✅ Audit done: 0 micro-boxes removed / 5051 total

🚀 RESUMING Phase 3 Training from /content/drive/MyDrive/RoadEye_V4_Final_Weights/v4_native_audit_run/weights/last.pt...
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=35, cls=0.6, cls_pw=0.0, compile=False, conf=None, copy_paste=0.15, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/RoadEye-4/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.4, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.008, lrf=0.1